In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
from pathlib import Path

In [ ]:
PROFILES_DIR  = "path_to_extracted_demographics" 
EXTRACTED_SIT_DIR = "path_to_extracted_situations"
PLOT_DIR = "path_to_save_plots"
PLOT_FILENAME = "plot_name.png"  # Specify the desired filename for the plot

In [ ]:
output_dir = Path(PLOT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
EMBED_MODEL      = "Qwen/Qwen3-Embedding-8B"   
EMBED_BATCH_SIZE = 16                            
EMBED_DIM        = None                          

TASK_INSTRUCTION = (
    "Given a counseling client's profile and presenting situation, "
    "retrieve the profile and situation describing the same client."
)
QUERY_PROMPT = f"Instruct: {TASK_INSTRUCTION}\nQuery:"

model = SentenceTransformer(
    EMBED_MODEL,
    truncate_dim=EMBED_DIM,
)

def embed_texts(texts, is_query):
    emb = model.encode(
        texts,
        prompt=QUERY_PROMPT if is_query else None,
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=False,
    )
    return np.asarray(emb, dtype=np.float32)
                  
SITUATION_KEY = "situation of the client"    
_DROP_VALUES = {"cannot be identified", "not enough information",
                "unknown", "n/a", "none", ""}

def standardize_profile(profile):
    if not profile:
        return {}
    out = {}
    for k, v in profile.items():
        if v is None:
            continue
        if isinstance(v, str) and v.strip().lower() in _DROP_VALUES:
            continue
        out[k] = v
    return out

def load_session_profiles(session_num, profiles_dir=PROFILES_DIR):
    path = os.path.join(profiles_dir, f"session_{session_num}.json")
    if not os.path.exists(path):
        return None, None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    real      = standardize_profile(data.get("actual profile", {}))
    extracted = standardize_profile(data.get("synthetic profile", {}))
    return real, extracted

def format_profile_situation(profile, situation):
    if profile:
        attrs = "; ".join(f"{k}: {v}" for k, v in profile.items())
    else:
        attrs = "(no identifiable attributes)"
    situation = (situation or "").strip()
    return f"Profile: {attrs}\nSituation: {situation}"

PROFILE_ATTRS = ["name", "gender", "age", "occupation", "marital status"]

def real_profile_from_parquet(parquet_idx):
    prof = json.loads(profiles[parquet_idx])
    return standardize_profile({k: prof[k] for k in PROFILE_ATTRS if k in prof})

In [ ]:
with open("../valid_indices.json", "r") as f:
    valid_indices = json.load(f)

In [ ]:
with open(EXTRACTED_SIT_DIR, 'r') as file:
    situation_1 = json.load(file)

In [ ]:
df = pd.read_parquet("../eeyore-data.parquet")

In [7]:
profiles = df['profile'].tolist()
ids = df['id_source'].tolist()

In [8]:
reference_text_list = []   # extracted profile + extracted situation (reference)
member_text_list    = []   # real profile + real situation (label 1)
session_list        = []
id_list             = []

In [ ]:
for i in range(len(profiles)):
    if ("session_" + str(i)) in situation_1.keys() and (i - 1) in valid_indices:
        # extracted ("synthetic") profile from session_<i>.json;
        # real profile from the parquet demographic fields.
        _, extracted_profile = load_session_profiles(i)
        if extracted_profile is None:        # session_<i>.json missing -> skip comparison
            continue
        real_profile = real_profile_from_parquet(i - 1)

        profile_i           = json.loads(profiles[i - 1])
        extracted_situation = situation_1["session_" + str(i)]
        real_situation      = profile_i[SITUATION_KEY]

        reference_text_list.append(
            format_profile_situation(extracted_profile, extracted_situation))
        member_text_list.append(
            format_profile_situation(real_profile, real_situation))
        session_list.append("session_" + str(i))
        id_list.append(ids[i - 1])

In [ ]:
embeddings_situation_reference = embed_texts(reference_text_list, is_query=False)

In [ ]:
embeddings_situation_member = embed_texts(member_text_list, is_query=True)

### Non-member construction

In [ ]:
with open("situation_non_member_valid_dedup_2_th_100.json", "r", encoding="utf-8") as f:
    situation_non_member_list = json.load(f)

with open("valid_non_member_indices_dedup_2_th_100.json", "r", encoding="utf-8") as f:
    non_member_indices = json.load(f)   # parquet row indices, aligned with the list above

assert len(non_member_indices) == len(situation_non_member_list), \
    "non-member indices and situations are not aligned"

def parquet_situation(parquet_idx):
    """Situation string for a parquet row, via the same path the rest of the code uses."""
    return json.loads(profiles[parquet_idx])[SITUATION_KEY]
mismatches = []
for pos, (idx, stored) in enumerate(zip(non_member_indices, situation_non_member_list)):
    fresh = parquet_situation(idx)
    if (fresh or "").strip() != (stored or "").strip():
        mismatches.append((pos, idx))

print(f"checked {len(non_member_indices)} non-members; {len(mismatches)} mismatches")
if mismatches:
    pos, idx = mismatches[0]
    print(f"first mismatch at list position {pos}, parquet index {idx}")
    print("  parquet:", repr(parquet_situation(idx)[:160]))
    print("  stored :", repr(situation_non_member_list[pos][:160]))
assert not mismatches, f"{len(mismatches)} non-member situations do not match the parquet"

In [16]:
non_member_text_list = [
    format_profile_situation(real_profile_from_parquet(idx), situation)
    for idx, situation in zip(non_member_indices, situation_non_member_list)
]

In [ ]:
embeddings_situation_non_member = embed_texts(non_member_text_list, is_query=True)

In [18]:
embeddings_ref = normalize(embeddings_situation_reference)
embeddings_mem = normalize(embeddings_situation_member)
embeddings_non_mem = normalize(embeddings_situation_non_member)

In [ ]:
def max_similarity_scores(embeddings_query, embeddings_ref, batch_size=None):
    """
    Compute max cosine similarity between each query embedding
    and all reference embeddings.

    Returns:
        scores: (N_query,)
    """

    if batch_size is None:
        sim_matrix = embeddings_query @ embeddings_ref.T
        scores = sim_matrix.max(axis=1)
        return scores

    scores = []

    for i in range(0, len(embeddings_query), batch_size):
        batch = embeddings_query[i:i+batch_size]
        sim = batch @ embeddings_ref.T
        scores.append(sim.max(axis=1))

    return np.concatenate(scores)

In [20]:
# ---- Compute scores ----
scores_member = max_similarity_scores(embeddings_mem, embeddings_ref)
scores_non_member = max_similarity_scores(embeddings_non_mem, embeddings_ref)

In [21]:
# Combine scores; keep as a numpy array so thresholding works downstream
all_scores = scores_member.tolist() + scores_non_member.tolist()

# Label list: members = 1, non-members = 0
labels = [1] * len(scores_member) + [0] * len(scores_non_member)

In [ ]:
fpr, tpr, thresholds = roc_curve(labels, all_scores)
roc_auc = auc(fpr, tpr)
print(f"ROC AUC: {roc_auc:.4f}")

In [ ]:
# ---- TPR at specific low FPR thresholds ----
target_fprs = [0.1, 0.05, 0.01]

print(f"{'FPR Target':<15} {'Actual FPR':<15} {'TPR':<10} {'Threshold':<12}")
print("-" * 55)
for target in target_fprs:
    # Find the index where fpr <= target (closest without exceeding)
    valid_idx = np.where(fpr <= target)[0]
    if len(valid_idx) == 0:
        print(f"{target:<15.2f} {'N/A':<15} {'N/A':<10} {'N/A':<12}")
        continue
    idx = valid_idx[-1]  # largest fpr that is still <= target
    print(f"{target:<15.2f} {fpr[idx]:<15.4f} {tpr[idx]:<10.4f} {thresholds[idx]:<12.4f}")

In [ ]:
# ---- Zoomed ROC Curve at low FPR region (log scale) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Full ROC for reference ---
ax1 = axes[0]
ax1.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
for target in [0.1, 0.05, 0.01]:
    valid_idx = np.where(fpr <= target)[0]
    if len(valid_idx) == 0:
        continue
    idx = valid_idx[-1]
    ax1.plot(fpr[idx], tpr[idx], 'o', markersize=7,
             label=f'FPR≤{target}: TPR={tpr[idx]:.3f}')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('Full ROC Curve')
ax1.legend(loc='lower right', fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Right: Zoomed ROC with log-scale FPR ---
ax2 = axes[1]
# Filter to low FPR region (fpr > 0 to allow log scale)
mask = (fpr > 0) & (tpr > 0)
fpr_log = fpr[mask]
tpr_log = tpr[mask]

ax2.plot(fpr_log, tpr_log, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
for target in [0.1, 0.05, 0.01]:
    valid_idx = np.where(fpr <= target)[0]
    if len(valid_idx) == 0:
        continue
    idx = valid_idx[-1]
    if fpr[idx] > 0:
        ax2.axvline(x=fpr[idx], color='gray', linestyle=':', alpha=0.6, lw=1)
        ax2.plot(fpr[idx], tpr[idx], 'o', markersize=8,
                 label=f'FPR≤{target}: TPR={tpr[idx]:.3f}')

ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim([0.005, 0.15])  # zoom into low FPR region
ax2.set_xlabel('False Positive Rate (log scale)')
ax2.set_ylabel('True Positive Rate (log scale)')
ax2.set_title('Zoomed ROC Curve — Low FPR Region (Log-Log Scale)')
ax2.legend(loc='lower right', fontsize=8)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR+PLOT_FILENAME, dpi=300, bbox_inches='tight')
plt.show()